# Visual Feature Extraction — MELD

Two extraction streams per utterance:

**Stream 1 — Semantic visual (GPU), temporal 3-segment pooling:**
- CLIP ViT-L/14 → `video_clip_temporal_{split}.pt`    — shape `(3, 768)`  per utterance (begin/mid/end)
- SigLIP 2      → `video_siglip2_temporal_{split}.pt` — shape `(3, 1152)` per utterance (begin/mid/end)

Each slot = mean of 3 frames from that temporal region of the clip.

**Stream 2 — Facial Action Units (GPU/CPU):**
- OpenFace 3.0 (`openface-test` package) → `video_openface_au_{split}.pt` — shape `(N_AU,)` per utterance

Output dir: `Dataset/Processed/MELD/features/`

---

**One-time installation:**
```bash
conda activate hopeful
pip install git+https://github.com/openai/CLIP.git
pip install openface-test && openface download        # weights → ~/.cache/openface
```

In [ ]:
import os
import sys
import cv2
import tempfile
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

In [ ]:
PROJECT_ROOT = Path("/mnt/Work/ML/Thesis/BMVC/Hopeful")
DATA_ROOT    = PROJECT_ROOT / "Dataset/Processed/MELD"
FEATURES_DIR = DATA_ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

FRAME_SAMPLE_FPS  = 2     # sample 2 frames/sec per clip
BATCH_SIZE_FRAMES = 32    # GPU batch size for CLIP / SigLIP 2
MAX_FRAMES        = 60    # cap — mirrors 30s audio truncation guard
AU_CONF_THRESHOLD = 0.5   # RetinaFace vis_threshold; exclude low-confidence faces
SPLITS            = ["train", "dev", "test"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device   : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

In [ ]:
labels = pd.read_csv(DATA_ROOT / "labels.csv")
print(f"Total utterances : {len(labels)}")
print(f"Split counts:\n{labels['split'].value_counts()}")

In [ ]:
def sample_frames(video_path: Path, fps: float = FRAME_SAMPLE_FPS,
                  max_frames: int = MAX_FRAMES) -> list[np.ndarray]:
    """Sample frames uniformly at `fps` from video. Returns list of BGR uint8 arrays."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    interval = max(1, int(round(src_fps / fps)))
    frames, idx = [], 0
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % interval == 0:
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

## 1. CLIP ViT-L/14 Extraction

In [ ]:
import clip

CLIP_MODEL_NAME = "ViT-L/14"
clip_model, clip_preprocess = clip.load(CLIP_MODEL_NAME, device=DEVICE)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad_(False)
CLIP_DIM = clip_model.visual.output_dim
N_FRAMES_PER_SEG = 3
print(f"CLIP loaded : {CLIP_MODEL_NAME}  |  dim={CLIP_DIM}  |  frames/segment={N_FRAMES_PER_SEG}")

def temporal_segments(frames, n=N_FRAMES_PER_SEG):
    """Split frames into (beginning, middle, end) lists of at most n frames each."""
    total = len(frames)
    if total == 0:
        return [], [], []
    mid    = total // 2
    half_n = n // 2
    beg     = frames[:n]
    mid_seg = frames[max(0, mid - half_n): max(0, mid - half_n) + n]
    end     = frames[max(0, total - n):]
    return beg, mid_seg, end

In [ ]:
def extract_clip(frames: list[np.ndarray]) -> np.ndarray:
    """Temporal 3-segment CLIP embeddings → (3, CLIP_DIM)."""
    if not frames:
        return np.zeros((3, CLIP_DIM), dtype=np.float32)
    vecs = []
    for seg in temporal_segments(frames):
        if not seg:
            vecs.append(np.zeros(CLIP_DIM, dtype=np.float32))
            continue
        pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in seg]
        tensors = torch.stack([clip_preprocess(img) for img in pil_frames]).to(DEVICE)
        with torch.no_grad():
            f = clip_model.encode_image(tensors).float()
            f = f / f.norm(dim=-1, keepdim=True)
        vecs.append(f.cpu().numpy().mean(axis=0))
    return np.stack(vecs)  # (3, CLIP_DIM)


all_skipped_clip: dict[str, list[str]] = {}

for split in SPLITS:
    out_path = FEATURES_DIR / f"video_clip_temporal_{split}.pt"
    if out_path.exists():
        print(f"[{split}] already exists — skipping"); continue

    split_df = labels[labels["split"] == split].reset_index(drop=True)
    print(f"\n--- CLIP temporal  {split}  ({len(split_df)} utterances) ---")
    features: dict[str, np.ndarray] = {}
    skipped:  list[str] = []

    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"CLIP {split}"):
        vid_path = DATA_ROOT / row["video_path"]
        if not vid_path.exists() or row["status"] != "ok":
            skipped.append(row["clip_name"]); continue
        frames = sample_frames(vid_path)
        if not frames:
            skipped.append(row["clip_name"]); continue
        features[row["clip_name"]] = extract_clip(frames)

    torch.save(features, out_path)
    print(f"Saved   : {out_path}  ({out_path.stat().st_size/1e6:.1f} MB, {len(features)} entries)")
    if skipped: print(f"Skipped : {skipped}")
    all_skipped_clip[split] = skipped

## 2. SigLIP 2 Extraction

In [ ]:
from transformers import AutoProcessor, AutoModel as HFAutoModel

SIGLIP2_MODEL_NAME = "google/siglip2-so400m-patch14-384"
siglip2_processor  = AutoProcessor.from_pretrained(SIGLIP2_MODEL_NAME)
siglip2_model      = HFAutoModel.from_pretrained(SIGLIP2_MODEL_NAME)
siglip2_model.eval()
for p in siglip2_model.parameters():
    p.requires_grad_(False)
siglip2_model = siglip2_model.to(DEVICE)
SIGLIP2_DIM = siglip2_model.config.vision_config.hidden_size
print(f"SigLIP 2 loaded : {SIGLIP2_MODEL_NAME}  |  dim={SIGLIP2_DIM}")

In [ ]:
def extract_siglip2(frames: list[np.ndarray]) -> np.ndarray:
    """Temporal 3-segment SigLIP 2 embeddings → (3, SIGLIP2_DIM)."""
    if not frames:
        return np.zeros((3, SIGLIP2_DIM), dtype=np.float32)
    vecs = []
    for seg in temporal_segments(frames):
        if not seg:
            vecs.append(np.zeros(SIGLIP2_DIM, dtype=np.float32))
            continue
        pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in seg]
        inputs = siglip2_processor(images=pil_frames, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = siglip2_model.vision_model(**inputs)
        vecs.append(out.pooler_output.cpu().float().numpy().mean(axis=0))
    return np.stack(vecs)  # (3, SIGLIP2_DIM)


all_skipped_siglip2: dict[str, list[str]] = {}

for split in SPLITS:
    out_path = FEATURES_DIR / f"video_siglip2_temporal_{split}.pt"
    if out_path.exists():
        print(f"[{split}] already exists — skipping"); continue

    split_df = labels[labels["split"] == split].reset_index(drop=True)
    print(f"\n--- SigLIP 2 temporal  {split}  ({len(split_df)} utterances) ---")
    features: dict[str, np.ndarray] = {}
    skipped:  list[str] = []

    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"SigLIP2 {split}"):
        vid_path = DATA_ROOT / row["video_path"]
        if not vid_path.exists() or row["status"] != "ok":
            skipped.append(row["clip_name"]); continue
        frames = sample_frames(vid_path)
        if not frames:
            skipped.append(row["clip_name"]); continue
        features[row["clip_name"]] = extract_siglip2(frames)

    torch.save(features, out_path)
    print(f"Saved   : {out_path}  ({out_path.stat().st_size/1e6:.1f} MB, {len(features)} entries)")
    if skipped: print(f"Skipped : {skipped}")
    all_skipped_siglip2[split] = skipped

## 3. OpenFace 3.0 — Action Unit Extraction

Requires `pip install openface-test && openface download` (weights → `~/.cache/openface`).

If weights landed elsewhere (e.g. cloned repo), set `OF3_WEIGHTS_DIR` below.

AUs detected: 18 FACS AU intensities (continuous 0–5 scale), mean-pooled over frames where face confidence ≥ `AU_CONF_THRESHOLD`.  
Zero vector stored for clips where no face is detected (flagged in `zero_face` list).

In [ ]:
# Weights downloaded by `openface download` land here by default.
# Override if you cloned the repo and put weights elsewhere:
#   OF3_WEIGHTS_DIR = Path("/path/to/OpenFace-3.0/weights")
OF3_WEIGHTS_DIR = Path.home() / ".cache" / "openface"

from openface.face_detection import FaceDetector
from openface.multitask_model import MultitaskPredictor

of3_face = FaceDetector(
    model_path=str(OF3_WEIGHTS_DIR / "Alignment_RetinaFace.pth"),
    device=str(DEVICE),
    vis_threshold=AU_CONF_THRESHOLD,
)
of3_pred = MultitaskPredictor(
    model_path=str(OF3_WEIGHTS_DIR / "MTL_backbone.pth"),
    device=str(DEVICE),
)
print("OpenFace 3.0 models loaded")

In [ ]:
def extract_au_of3(frames: list[np.ndarray]) -> tuple[np.ndarray | None, int]:
    """
    Run OpenFace 3.0 AU detection on sampled frames.
    Saves each frame to a temp file (FaceDetector.get_face requires a path).
    Returns (mean_au_vector, n_detected) or (None, 0) if no face found in any frame.
    """
    au_list = []
    with tempfile.TemporaryDirectory() as tmpdir:
        for i, frame in enumerate(frames):
            frame_path = os.path.join(tmpdir, f"f{i:04d}.jpg")
            cv2.imwrite(frame_path, frame)
            cropped_face, dets = of3_face.get_face(frame_path)
            if cropped_face is None:
                continue
            _, _, au_output = of3_pred.predict(cropped_face)
            au_list.append(au_output.cpu().float().numpy().flatten())
    if not au_list:
        return None, 0
    stacked = np.stack(au_list)          # (n_frames, N_AU)
    return stacked.mean(axis=0), len(au_list)


all_skipped_au: dict[str, list[str]] = {}
zero_face_au:   dict[str, list[str]] = {}
AU_DIM: int | None = None              # determined from first successful detection

for split in SPLITS:
    out_path = FEATURES_DIR / f"video_openface_au_{split}.pt"
    if out_path.exists():
        print(f"[{split}] already exists — skipping"); continue

    split_df = labels[labels["split"] == split].reset_index(drop=True)
    print(f"\n--- OpenFace 3.0 AU  {split}  ({len(split_df)} utterances) ---")
    features:  dict[str, np.ndarray] = {}
    skipped:   list[str] = []
    zero_face: list[str] = []

    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"AU {split}"):
        vid_path = DATA_ROOT / row["video_path"]
        if not vid_path.exists() or row["status"] != "ok":
            skipped.append(row["clip_name"]); continue
        frames = sample_frames(vid_path)
        if not frames:
            skipped.append(row["clip_name"]); continue

        au_vec, n_det = extract_au_of3(frames)

        if au_vec is None:
            zero_face.append(row["clip_name"])
            if AU_DIM is not None:
                features[row["clip_name"]] = np.zeros(AU_DIM, dtype=np.float32)
            continue

        if AU_DIM is None:
            AU_DIM = len(au_vec)
            print(f"  AU dim detected: {AU_DIM}")
            for name in zero_face:   # back-fill earlier zero-face entries
                features[name] = np.zeros(AU_DIM, dtype=np.float32)

        features[row["clip_name"]] = au_vec.astype(np.float32)

    # back-fill any remaining zero-face entries
    if AU_DIM:
        for name in zero_face:
            if name not in features:
                features[name] = np.zeros(AU_DIM, dtype=np.float32)

    torch.save(features, out_path)
    n_zero = sum(1 for v in features.values() if v.sum() == 0)
    cov = len(features) - n_zero
    print(f"Saved   : {out_path}  ({out_path.stat().st_size/1e6:.1f} MB, {len(features)} entries)")
    print(f"Face coverage : {cov}/{len(features)}  ({cov/max(len(features),1)*100:.1f}%)")
    if skipped:    print(f"Skipped : {skipped}")
    if zero_face:  print(f"No face : {zero_face}")
    all_skipped_au[split] = skipped
    zero_face_au[split]   = zero_face

## 4. Verification

In [ ]:
import numpy as np

print("=== Verification ===\n")

expected_counts = {"train": 9989, "dev": 1109, "test": 2610}

for split in SPLITS:
    exp = expected_counts[split]
    print(f"[{split}]")
    for tag, expected_shape in [
        (f"video_clip_temporal_{split}",    (3, CLIP_DIM)),
        (f"video_siglip2_temporal_{split}", (3, SIGLIP2_DIM)),
        (f"video_openface_au_{split}",      (AU_DIM,)),
    ]:
        pt = FEATURES_DIR / f"{tag}.pt"
        if not pt.exists():
            print(f"  {tag}: MISSING"); continue
        d = torch.load(pt, weights_only=False)
        feat = next(iter(d.values()))
        all_feats = np.stack(list(d.values()))
        shape_ok = feat.shape == expected_shape
        print(f"  {tag}: count={len(d)}/{exp}  shape={feat.shape} "
              f"{'✓' if shape_ok else '✗ expected '+str(expected_shape)}  "
              f"nan={np.isnan(all_feats).any()}  inf={np.isinf(all_feats).any()}")
    print()